In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import os
from matplotlib import colors, rcParams
from pysteps.utils import transformation
import traceback

# Set global font sizes to 24
rcParams.update({
    'font.size': 24,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 24,
    'figure.titlesize': 24,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

def create_consistent_colormap(units="mm/h"):
    if units in ["mm/h", "mm"]:
        clevs = [0.08, 0.16, 0.25, 0.40, 0.63, 1, 1.6, 2.5, 4, 6.3, 10, 16, 25, 40, 63, 100, 160]
        color_list = [
            '#9c7e94', '#640064', '#AF00AF', '#DC00DC', '#3232C8',
            '#0064FF', '#009696', '#00C832', '#64FF00', '#96FF00',
            '#C8FF00', '#FFFF00', '#FFC800', '#FFA000', '#FF7D00', '#E11900'
        ]
    else:
        clevs = np.arange(10, 65, 5)
        color_list = plt.cm.jet(np.linspace(0, 1, len(clevs)))
    
    cmap = colors.LinearSegmentedColormap.from_list("pysteps_cmap", color_list, len(clevs)-1)
    cmap.set_over("darkred", 1)
    cmap.set_bad("white", alpha=0)
    cmap.set_under("none")
    norm = colors.BoundaryNorm(clevs, cmap.N)
    
    return cmap, norm, clevs

def plot_observation_data(root_path, output_dir):
    """Process and plot all observation files with custom colors and larger fonts"""
    print(f"Input directory: {root_path}")
    print(f"Output directory: {output_dir}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Use our custom colormap
    cmap, norm, clevs = create_consistent_colormap()
    
    metadata = {
        "unit": "mm/h",
        "transform": "dB",
        'x1': 107, 'x2': 108,
        'y1': -7.2, 'y2': -6.4,
        'yorigin': 'upper'
    }

    mat_files = [f for f in os.listdir(root_path) if f.endswith('.mat')]
    print(f"Found {len(mat_files)} MAT files to process")

    for filename in sorted(mat_files):
        file_path = os.path.join(root_path, filename)
        print(f"Processing: {filename}")
        
        try:
            mat_data = scipy.io.loadmat(file_path)
            
            if 'ZI' not in mat_data:
                print(f"Skipping {filename}: 'ZI' field not found")
                continue
            
            reflectivity = mat_data['ZI']
            reflectivity = reflectivity / 10000
            reflectivity[reflectivity == -1] = np.nan
            
            precip = transformation.dB_transform(
                reflectivity[np.newaxis, :, :], 
                threshold=0.1, 
                zerovalue=-15.0, 
                inverse=True
            )[0][0]
            
            # Create plot with larger fonts
            fig, ax = plt.subplots(figsize=(12, 10), facecolor='white')
            ax.set_facecolor('white')
            
            im = ax.imshow(
                precip,
                extent=[metadata['x1'], metadata['x2'], metadata['y1'], metadata['y2']],
                origin='upper',
                cmap=cmap,
                norm=norm
            )
            
            # Customize colorbar with larger fonts
            cbar = plt.colorbar(im, ax=ax, extend='max', pad=0.02)
            cbar.set_label('Precipitation (mm/h)', fontsize=24)
            cbar.ax.tick_params(labelsize=24)
            
            # Extract time from filename
            time_str = "Unknown"
            if '_' in filename:
                time_part = filename.split('_')[-1].replace('.mat', '')
                if len(time_part) == 6 and time_part.isdigit():
                    time_str = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:6]}"
            
            # Customize title and labels with larger fonts
            ax.set_title(f"Observed Precipitation - {time_str}", 
                        fontsize=24, pad=20, weight='bold')
            ax.set_xlabel('Longitude', fontsize=24)
            ax.set_ylabel('Latitude', fontsize=24)
            
            # Adjust ticks with larger fonts
            ax.tick_params(axis='both', which='major', labelsize=26)
            
            output_filename = f"obs_{filename.replace('.mat', '.png')}"
            output_path = os.path.join(output_dir, output_filename)
            plt.savefig(output_path, bbox_inches='tight', dpi=150, facecolor='white')
            plt.close()
            print(f"Saved: {output_path}")
            
        except Exception as e:
            print(f"Error processing {filename}:")
            traceback.print_exc()

# Example usage with complete paths
root_path = r"D:/College/MENEA/FIXED_DATA/RUN_TEST/20210201/"
output_dir = r"D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/"

plot_observation_data(root_path, output_dir)
print("Processing complete!")

Input directory: D:/College/MENEA/FIXED_DATA/RUN_TEST/20210201/
Output directory: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/
Found 30 MAT files to process
Processing: new_20210201_171810.mat
Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/obs_new_20210201_171810.png
Processing: new_20210201_172011.mat
Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/obs_new_20210201_172011.png
Processing: new_20210201_172211.mat
Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/obs_new_20210201_172211.png
Processing: new_20210201_172412.mat
Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/obs_new_20210201_172412.png
Processing: new_20210201_172610.mat
Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/obs_new_20210201_172610.png
Processing: new_20210201_172811.mat
Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20210201/observations/obs_new_20210201_172811.png
Processing: new_20210201_173011.mat
Saved: D:/College/MENEA/FI

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import os
from matplotlib import colors, rcParams
from datetime import datetime
import cartopy.crs as crs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from pysteps.utils import transformation
import traceback

# Set font sizes to match Script 2 exactly
rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

def create_precip_colormap():
    """Create precipitation colormap using EXACT Script 1 colors"""
    clevs = [0.08,0.16,0.25,0.40,0.63,1,1.6,2.5,4,6.3,10,16,25,40,63,100,160]
    color_list = [
        '#9c7e94','#640064','#AF00AF','#DC00DC','#3232C8',
        '#0064FF','#009696','#00C832','#64FF00','#96FF00',
        '#C8FF00','#FFFF00','#FFC800','#FFA000','#FF7D00','#E11900'
    ]
    
    cmap = colors.LinearSegmentedColormap.from_list(
        "pysteps_precip", color_list, len(clevs)-1)
    cmap.set_over("darkred", 1)
    cmap.set_bad("gray", alpha=0.5)
    cmap.set_under("none")
    norm = colors.BoundaryNorm(clevs, cmap.N)
    
    return cmap, norm, clevs

def load_and_process_data(root_path):
    """Load and process data using Script 1's method"""
    R = []
    times = []
    filenames = []
    
    for filename in sorted(os.listdir(root_path)):
        if filename.endswith('.mat'):
            file_path = os.path.join(root_path, filename)
            
            try:
                mat_data = scipy.io.loadmat(file_path)
                
                if 'ZI' not in mat_data:
                    continue
                
                reflectivity = mat_data['ZI'] / 10000
                reflectivity[reflectivity == -1] = np.nan
                
                # Transform using Script 1's dB transform
                precip = transformation.dB_transform(
                    reflectivity[np.newaxis, :, :], 
                    threshold=0.1, 
                    zerovalue=-15.0, 
                    inverse=True
                )[0][0]
                
                R.append(precip)
                filenames.append(filename)
                
                # Extract time
                if '_' in filename:
                    time_part = filename.split('_')[-1].replace('.mat', '')
                    if len(time_part) == 6 and time_part.isdigit():
                        times.append(f"{time_part[:2]}:{time_part[2:4]}")
                        
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
                continue
    
    if not R:
        raise ValueError("No valid data found")
    
    metadata = {
        'x1': 107, 'x2': 108,
        'y1': -7.1, 'y2': -6.4,  # Using Script 2's y-range
        'yorigin': 'upper'
    }
    
    return np.array(R), times, filenames, metadata

def plot_observations_script2_style():
    # Configuration
    date_str = "20221109"
    date = datetime.strptime(date_str, "%Y%m%d")
    root_path = r"D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/"
    output_dir = r"D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/"
    os.makedirs(output_dir, exist_ok=True)
    
    # Load and process data
    R, times, filenames, metadata = load_and_process_data(root_path)
    
    # Get Script 1's colormap
    cmap, norm, clevs = create_precip_colormap()
    
    # Select 6 time steps as in Script 2
    selected_indices = [4, 9, 14, 19, 24, 29]
    selected_data = [R[i] for i in selected_indices]
    selected_times = [times[i] if i < len(times) else "Unknown" for i in selected_indices]
    
    # Create figure with Script 2's dimensions
    fig = plt.figure(figsize=(32, 18))  # Increased height to accommodate both colorbars
    
    # Adjust subplot spacing to make room for both colorbars
    plt.subplots_adjust(
        left=0.08, right=0.92, 
        bottom=0.18, top=0.85,  # Adjusted bottom and top margins
        wspace=0.25, hspace=0.7  # Increased hspace to separate rows
    )
    
    # Create subplots
    axs = []
    for i in range(6):
        ax = fig.add_subplot(2, 3, i+1, projection=crs.PlateCarree())
        axs.append(ax)
    
    # Plot each observation
    for i, (data, time_str) in enumerate(zip(selected_data, selected_times)):
        plot_data = np.where(np.isfinite(data), data, 0)
        
        im = axs[i].imshow(
            plot_data,
            extent=[metadata['x1'], metadata['x2'], metadata['y1'], metadata['y2']],
            cmap=cmap,
            norm=norm,
            transform=crs.PlateCarree(),
            origin='upper',
            aspect='auto'
        )
        
        # Configure gridlines with custom font size for lat/lon labels
        gl = axs[i].gridlines(
            draw_labels=True,
            linestyle='--',
            alpha=0.5,
            linewidth=1.0,
            xpadding=8,  # Increased padding
            ypadding=8)
        
        # Set ticks like Script 2
        gl.xlocator = plt.FixedLocator([107.2, 107.4, 107.6, 107.8, 108.0])
        gl.ylocator = plt.FixedLocator([-7.0, -6.8, -6.6])
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        
        # Set LATITUDE AND LONGITUDE LABEL FONT SIZE TO 22
        gl.xlabel_style = {'size': 32}  # Longitude labels
        gl.ylabel_style = {'size': 32}  # Latitude labels
        
        # Label positioning EXACTLY like Script 2
        if i < 3:  # Top row
            gl.bottom_labels = True
            gl.top_labels = False
            if i == 1:  # Middle top - adjust longitude labels up
                gl.xlabels_bottom = False
                gl.xlabels_top = True
        else:      # Bottom row
            gl.top_labels = False
            gl.bottom_labels = True
            
        if i % 3 == 0:  # Left column
            gl.left_labels = True
            gl.right_labels = False
        else:            # Middle and right columns
            gl.left_labels = False
            gl.right_labels = False
        
        # Set title (keeping original size)
        axs[i].set_title(f"Observation at {time_str}", fontsize=28, pad=10)
        axs[i].set_xlim(107, 108)
        axs[i].set_ylim(-7.1, -6.4)
        axs[i].coastlines(resolution='10m', color='black', linewidth=0.5)
    
    # Add second colorbar below row 2 (original position)
    cbar_ax2 = fig.add_axes([0.25, 0.1, 0.5, 0.015])  # Adjusted position and height
    cbar2 = fig.colorbar(im, cax=cbar_ax2, orientation='horizontal', extend='max')
    cbar2.set_label('Precipitation intensity [mm/h]', fontsize=28, labelpad=10)
    cbar2.ax.tick_params(labelsize=28)
    
    # Save figure
    output_path = os.path.join(output_dir, "observations_script2_layout_with_dual_cb_adjusted.png")
    plt.savefig(output_path, bbox_inches='tight', dpi=300, facecolor='white')
    print(f"Saved: {output_path}")
    plt.close()

if __name__ == "__main__":
    plot_observations_script2_style()

Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/observations_script2_layout_with_dual_cb_adjusted.png


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import os
from matplotlib import colors, rcParams
from datetime import datetime
import cartopy.crs as crs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from pysteps.utils import transformation

# Set font sizes to match exactly (adjusted for single plot)
rcParams.update({
    'font.size': 28,  # Larger for single plot visibility
    'axes.titlesize': 28,
    'axes.labelsize': 28,
    'xtick.labelsize': 28,
    'ytick.labelsize': 28,
    'figure.titlesize': 28,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white'
})

def create_precip_colormap():
    """Create precipitation colormap using EXACT Script 1 colors"""
    clevs = [0.08,0.16,0.25,0.40,0.63,1,1.6,2.5,4,6.3,10,16,25,40,63,100,160]
    color_list = [
        '#9c7e94','#640064','#AF00AF','#DC00DC','#3232C8',
        '#0064FF','#009696','#00C832','#64FF00','#96FF00',
        '#C8FF00','#FFFF00','#FFC800','#FFA000','#FF7D00','#E11900'
    ]
    
    cmap = colors.LinearSegmentedColormap.from_list(
        "pysteps_precip", color_list, len(clevs)-1)
    cmap.set_over("darkred", 1)
    cmap.set_bad("gray", alpha=0.5)
    cmap.set_under("none")
    norm = colors.BoundaryNorm(clevs, cmap.N)
    
    return cmap, norm, clevs

def load_and_process_single_file(root_path, file_index=15):
    """Load and process specific file using Script 1's method"""
    filenames = sorted([f for f in os.listdir(root_path) if f.endswith('.mat')])
    if file_index >= len(filenames):
        raise ValueError(f"File index {file_index} out of range (max {len(filenames)-1})")
    
    filename = filenames[file_index]
    file_path = os.path.join(root_path, filename)
    
    try:
        mat_data = scipy.io.loadmat(file_path)
        
        if 'ZI' not in mat_data:
            raise ValueError("ZI field not found in mat file")
        
        reflectivity = mat_data['ZI'] / 10000
        reflectivity[reflectivity == -1] = np.nan
        
        # Transform using Script 1's dB transform
        precip = transformation.dB_transform(
            reflectivity[np.newaxis, :, :], 
            threshold=0.1, 
            zerovalue=-15.0, 
            inverse=True
        )[0][0]
        
        # Extract time
        time_str = "Unknown"
        if '_' in filename:
            time_part = filename.split('_')[-1].replace('.mat', '')
            if len(time_part) == 6 and time_part.isdigit():
                time_str = f"{time_part[:2]}:{time_part[2:4]}"
                
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        raise
    
    metadata = {
        'x1': 107, 'x2': 108,
        'y1': -7.1, 'y2': -6.4,
        'yorigin': 'upper'
    }
    
    return precip, time_str, filename, metadata

def plot_single_observation():
    # Configuration
    date_str = "20221109"
    date = datetime.strptime(date_str, "%Y%m%d")
    root_path = r"D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/"
    output_dir = r"D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/"
    os.makedirs(output_dir, exist_ok=True)
    
    # File index to plot (16th file = index 15)
    file_index = 15
    
    # Load and process data
    data, time_str, filename, metadata = load_and_process_single_file(root_path, file_index)
    
    # Get Script 1's colormap
    cmap, norm, clevs = create_precip_colormap()
    
    # Create single figure (adjusted size for better visibility)
    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_subplot(1, 1, 1, projection=crs.PlateCarree())
    
    # Plot the observation
    plot_data = np.where(np.isfinite(data), data, 0)
    im = ax.imshow(
        plot_data,
        extent=[metadata['x1'], metadata['x2'], metadata['y1'], metadata['y2']],
        cmap=cmap,
        norm=norm,
        transform=crs.PlateCarree(),
        origin='upper',
        aspect='auto'
    )
    
    # Configure gridlines - DISABLE TOP AND RIGHT LABELS
    gl = ax.gridlines(
        draw_labels=True,
        linestyle='--',
        alpha=0.5,
        linewidth=1.0)
    
    # Set ticks like Script 2
    gl.xlocator = plt.FixedLocator([107.2, 107.4, 107.6, 107.8, 108.0])
    gl.ylocator = plt.FixedLocator([-7.0, -6.8, -6.6])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # DISABLE TOP AND RIGHT LABELS
    gl.top_labels = False
    gl.right_labels = False
    
    # Set label styles
    gl.xlabel_style = {'size': 32}  # Longitude labels
    gl.ylabel_style = {'size': 32}  # Latitude labels
    
    
    # Set title and limits
    ax.set_title(f"Observation at {time_str}", fontsize=28, pad=20)
    ax.set_xlim(107, 108)
    ax.set_ylim(-7.1, -6.4)
    ax.coastlines(resolution='10m', color='black', linewidth=1.0)
    
    # Save figure
    output_path = os.path.join(output_dir, f"single_observation_file_{file_index+1}.eps")
    plt.savefig(output_path, bbox_inches='tight', dpi=300, facecolor='white')
    print(f"Saved: {output_path}")
    plt.close()

if __name__ == "__main__":
    plot_single_observation()

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved: D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/single_observation_file_16.eps
